# Analyzing data from Task 1 

Note to self: trying to run `cmdstanpy` on windows is quite painful ... let's try to keep the workflow mostly in MacOS ... 

In [2]:
import polars as pl
import pandas as pd
import pymc as pm
import arviz as az
import arviz_plots as azp
import os 
import shutil 
import cmdstanpy 
from cmdstanpy import CmdStanModel
from cmdstanpy import cmdstan_path
print(f'cmdstan is installed at {cmdstan_path()}')

cmdstan is installed at /Users/shenglong/.cmdstan/cmdstan-2.38.0


In [3]:
# read in data 
task1_df = pl.read_csv("../../data/processed/task1.csv")
task1_df.head()

,participantId,task,id,rsp_time,status,data.select.x,data.select.y,pixel.select.x,pixel.select.y,param.mu,param.sigma,param.lambda,param.p,param.q,pixelToMM,dist_to_screen,data.select.left_area,data.ans.x,pixel.med.x,pixel.mod.x,phy.select.x,phy.med.x,phy.mod.x,va.select.x,va.med.x,va.mod.x
i64,str,str,i64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""67f03733938aa3f0d17ce8a8""","""task1""",1,8.22,"""completed""",0.34,0.218606,335.69,323.650744,0.41,1.5,-0.01,3.69,16.55,4.6,668.258963,"""0.489697578893479""",0.387128,338.211338,339.435,62.106522,62.654639,62.920652,5.321116,5.368009,5.390766
2,"""67f03733938aa3f0d17ce8a8""","""task1""",2,5.467,"""completed""",0.09,0.310524,322.315,287.34312,-0.65,1.17,0.53,2.92,5.89,4.6,668.258963,"""0.4692395323358""",0.189605,327.643861,282.725,59.198913,60.357361,50.592391,5.072333,5.171459,4.335665
3,"""67f03733938aa3f0d17ce8a8""","""task1""",3,5.354,"""completed""",0.34,0.521065,335.69,204.179468,0.42,0.62,-0.15,3.94,37.73,4.6,668.258963,"""0.53331408923348""",0.276061,332.26924,339.97,62.106522,61.362878,63.036957,5.321116,5.257493,5.400716
4,"""67f03733938aa3f0d17ce8a8""","""task1""",4,5.059,"""completed""",0.09,0.350077,322.315,271.719768,-0.89,1.05,0.79,2.6,27.22,4.6,668.258963,"""0.467448266910563""",0.183909,327.339129,269.885,59.198913,60.291115,47.801087,5.072333,5.165791,4.096666
5,"""67f03733938aa3f0d17ce8a8""","""task1""",5,5.606,"""completed""",-0.43,0.634068,294.495,159.543153,-0.37,0.51,-0.15,3.91,46.15,4.6,668.258963,"""0.536955412831699""",-0.488288,291.376613,297.705,53.151087,52.473177,53.848913,4.554715,4.496682,4.61445


Get the data ready for stan: 

In [38]:
numeric_id = task1_df.select(pl.col('participantId').cast(pl.Categorical).to_physical()).to_numpy().flatten()
# make sure the ids are not 0-indexed 
numeric_id += 1 

stan_data = {
    'N': task1_df.height, 
    'J': task1_df.select(pl.col('participantId')).n_unique(), 
    'id': numeric_id, 
    'x': task1_df.select('va.select.x').to_numpy().flatten(), 
    'x_med': task1_df.select('va.med.x').to_numpy().flatten(),
    'x_mod': task1_df.select('va.mod.x').to_numpy().flatten()
}

# stan_data

# Compile stan models 

## `task1.weighted.avg.stan` 

- centered parameterization 

In [39]:
task1_mix_stanfile = os.path.join('../stan_files/', "task1.weighted.avg.stan")
task1_model = CmdStanModel(stan_file=task1_mix_stanfile)

In [40]:
# use shutil to move things around
dest_dir = "../stan_bin_files"
shutil.move(task1_model.exe_file, os.path.join(dest_dir, task1_model.name))
task1_model = CmdStanModel(exe_file=os.path.join(dest_dir, task1_model.name))

In [44]:
# sample 
fit1 = task1_model.sample(
    data = stan_data, 
    chains=4, 
    parallel_chains=4, 
    iter_warmup=2000,
    iter_sampling=4000,
    adapt_delta=0.99, 
    show_console=False, 
)

12:50:08 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/6000 [00:00<?, ?it/s, (Warmup)]


chain 1:   2%|▏         | 100/6000 [00:01<01:23, 70.90it/s, (Warmup)]


chain 1:   3%|▎         | 200/6000 [00:01<00:46, 125.34it/s, (Warmup)]

chain 1:   5%|▌         | 300/6000 [00:01<00:30, 189.53it/s, (Warmup)]


chain 1:   7%|▋         | 400/6000 [00:02<00:23, 241.44it/s, (Warmup)]



chain 1:   8%|▊         | 500/6000 [00:02<00:21, 258.41it/s, (Warmup)]

chain 1:  10%|█         | 600/6000 [00:02<00:17, 315.29it/s, (Warmup)]

chain 1:  12%|█▏        | 700/6000 [00:02<00:14, 362.68it/s, (Warmup)]

chain 1:  13%|█▎        | 800/6000 [00:03<00:12, 404.64it/s, (Warmup)]

chain 1:  15%|█▌        | 900/6000 [00:03<00:11, 447.33it/s, (Warmup)]



chain 1:  22%|██▏       | 1300/6000 [00:04<00:11, 399.26it/s, (Warmup)]

chain 1:  25%|██▌       | 1500/6000 [00:04<00:10, 431.32it/s, (Warmup)]


chain 1:  27%|██▋       | 1600/6000 [00:05<00:10, 419.74it/s, (Warmup)]

chain 1:  


12:50:31 - cmdstanpy - INFO - CmdStan done processing.
12:50:31 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'task1.weighted.avg.stan', line 57, column 4 to column 47)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'task1.weighted.avg.stan', line 57, column 4 to column 47)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'task1.weighted.avg.stan', line 57, column 4 to column 47)
Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'task1.weighted.avg.stan', line 58, column 4 to column 67)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'task1.weighted.avg.stan', line 58, column 4 to column 67)
	Exception: task1x46weightedx46avg_model_namespace::log_prob: theta[21] is nan, but must be greater than or equal to 0.000000 (in 'task1.weighted.avg.stan', line 31, column 2 to column 36)
Excepti

12:50:32 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 2 had 2 divergent transitions (0.1%)
	Chain 3 had 710 divergent transitions (17.8%)
	Chain 4 had 34 divergent transitions (0.9%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


Note that some chains have failed to converge. 

In [43]:
fit1.diagnose()

"Checking sampler transitions treedepth.\nTreedepth satisfactory for all transitions.\n\nChecking sampler transitions for divergences.\n234 of 16000 (1.46%) transitions ended with a divergence.\nThese divergent transitions indicate that HMC is not fully able to explore the posterior distribution.\nTry increasing adapt delta closer to 1.\nIf this doesn't remove all divergences, try to reparameterize the model.\n\nChecking E-BFMI - sampler transitions HMC potential energy.\nE-BFMI satisfactory.\n\nRank-normalized split effective sample size satisfactory for all parameters.\n\nRank-normalized split R-hat values satisfactory for all parameters.\n\nProcessing complete.\n"

In [46]:
# Translate to InferenceData 

task1_idata = az.from_cmdstanpy(fit1)
task1_idata

Inference data with groups:
	> posterior
	> log_likelihood
	> sample_stats

In [47]:
# we could try to plot things ...? 

az.summary(task1_idata)

/Users/shenglong/Downloads/vis-decode-analysis/.venv/lib/python3.13/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/Users/shenglong/Downloads/vis-decode-analysis/.venv/lib/python3.13/site-packages/arviz/stats/diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_mod_mean,0.005,0.018,-0.027,0.039,0.000,0.000,6602.0,10367.0,1.00
mu_mod_sd,0.047,0.024,0.002,0.086,0.002,0.002,95.0,16.0,1.04
mu_med_mean,0.053,0.050,-0.045,0.146,0.001,0.001,3890.0,8165.0,1.01
mu_med_sd,0.130,0.053,0.024,0.222,0.003,0.001,325.0,762.0,1.02
log_sigma_mod_mean,-1.493,0.092,-1.674,-1.326,0.001,0.001,14076.0,11583.0,1.01
...,...,...,...,...,...,...,...,...,...
x_org[155],7.272,0.000,7.272,7.272,0.000,0.000,16000.0,16000.0,NaN
x_org[156],7.608,0.000,7.608,7.608,0.000,0.000,16000.0,16000.0,NaN
x_org[157],7.457,0.000,7.457,7.457,0.000,0.000,16000.0,16000.0,NaN
x_org[158],6.623,0.000,6.623,6.623,0.000,0.000,16000.0,16000.0,NaN


## `task1.weighted.avg.nonc.stan` 

- non-centered parameterization 

In [48]:
task1_mix_stanfile = os.path.join('../stan_files/', "task1.weighted.avg.nonc.stan")
task1_model = CmdStanModel(stan_file=task1_mix_stanfile)

12:52:57 - cmdstanpy - INFO - compiling stan file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.stan to exe file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc
12:53:03 - cmdstanpy - INFO - compiled model executable: /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc


In [49]:
# use shutil to move things around
dest_dir = "../stan_bin_files"
shutil.move(task1_model.exe_file, os.path.join(dest_dir, task1_model.name))
task1_model = CmdStanModel(exe_file=os.path.join(dest_dir, task1_model.name))

In [52]:
# sample 
fit2 = task1_model.sample(
    data = stan_data, 
    chains=4, 
    parallel_chains=4, 
    iter_warmup=2000,
    iter_sampling=4000,
    adapt_delta=0.95, 
    show_console=True,
)

12:54:20 - cmdstanpy - INFO - Chain [1] start processing
12:54:20 - cmdstanpy - INFO - Chain [2] start processing
12:54:20 - cmdstanpy - INFO - Chain [3] start processing
12:54:20 - cmdstanpy - INFO - Chain [4] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 4000
Chain [1] num_warmup = 2000
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [2] method = sample (Default)
Chain [2] sample
Chain [2] num_samples = 4000
Chain [2] num_warmup = 2000
Chain [2] save_warmup = false (Default)
Chain [2] thin = 1 (Default)
Chain [2] adapt
Chain [2] engaged = true (Default)
Chain [2] gamma = 0.05 (Default)
Chain [2] delta = 0.95
Chain [2] kappa = 0.75 (Default)
Chain [2] t0 = 10 (Default)
Chain [2] init_buffer = 75 (Default)
Chain [2] term_buffer = 50 (Default)
Chain [2] window = 25 (Default)
Chai

12:54:30 - cmdstanpy - INFO - Chain [2] done processing


Chain [4] Iteration: 5600 / 6000 [ 93%]  (Sampling)
Chain [2] Iteration: 5900 / 6000 [ 98%]  (Sampling)
Chain [1] Iteration: 4300 / 6000 [ 71%]  (Sampling)
Chain [2] Iteration: 6000 / 6000 [100%]  (Sampling)
Chain [2] 
Chain [2] Elapsed Time: 4.419 seconds (Warm-up)
Chain [2] 5.109 seconds (Sampling)
Chain [2] 9.528 seconds (Total)
Chain [2] 
Chain [4] Iteration: 5700 / 6000 [ 95%]  (Sampling)
Chain [2] 
Chain [2] 
Chain [3] Iteration: 4200 / 6000 [ 70%]  (Sampling)
Chain [4] Iteration: 5800 / 6000 [ 96%]  (Sampling)
Chain [1] Iteration: 4400 / 6000 [ 73%]  (Sampling)
Chain [3] Iteration: 4300 / 6000 [ 71%]  (Sampling)
Chain [4] Iteration: 5900 / 6000 [ 98%]  (Sampling)


12:54:30 - cmdstanpy - INFO - Chain [4] done processing


Chain [1] Iteration: 4500 / 6000 [ 75%]  (Sampling)
Chain [4] Iteration: 6000 / 6000 [100%]  (Sampling)
Chain [4] 
Chain [4] Elapsed Time: 4.364 seconds (Warm-up)
Chain [4] 5.536 seconds (Sampling)
Chain [4] 9.9 seconds (Total)
Chain [4] 
Chain [4] 
Chain [3] Iteration: 4400 / 6000 [ 73%]  (Sampling)
Chain [1] Iteration: 4600 / 6000 [ 76%]  (Sampling)
Chain [3] Iteration: 4500 / 6000 [ 75%]  (Sampling)
Chain [1] Iteration: 4700 / 6000 [ 78%]  (Sampling)
Chain [3] Iteration: 4600 / 6000 [ 76%]  (Sampling)
Chain [1] Iteration: 4800 / 6000 [ 80%]  (Sampling)
Chain [3] Iteration: 4700 / 6000 [ 78%]  (Sampling)
Chain [1] Iteration: 4900 / 6000 [ 81%]  (Sampling)
Chain [3] Iteration: 4800 / 6000 [ 80%]  (Sampling)
Chain [1] Iteration: 5000 / 6000 [ 83%]  (Sampling)
Chain [3] Iteration: 4900 / 6000 [ 81%]  (Sampling)
Chain [1] Iteration: 5100 / 6000 [ 85%]  (Sampling)
Chain [3] Iteration: 5000 / 6000 [ 83%]  (Sampling)
Chain [1] Iteration: 5200 / 6000 [ 86%]  (Sampling)
Chain [3] Iteration: 5

12:54:33 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] 
Chain [1] Elapsed Time: 4.198 seconds (Warm-up)
Chain [1] 8.705 seconds (Sampling)
Chain [1] 12.903 seconds (Total)
Chain [1] 
Chain [1] 
Chain [3] Iteration: 5900 / 6000 [ 98%]  (Sampling)


12:54:33 - cmdstanpy - INFO - Chain [3] done processing
12:54:33 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to line 76, column 126)
	Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to line 76, column 126)
	Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to line 76, column 126)
	Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to line 76, column 126)
	Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to line 76, column 126)
	Exception: normal_lpdf: Location parameter is nan, but must be finite! (in 'task1.weighted.avg.nonc.stan', line 75, column 4 to 

Chain [3] Iteration: 6000 / 6000 [100%]  (Sampling)
Chain [3] 
Chain [3] Elapsed Time: 4.491 seconds (Warm-up)
Chain [3] 8.733 seconds (Sampling)
Chain [3] 13.224 seconds (Total)
Chain [3] 
Chain [3] 
Chain [3] 


In [53]:
task1_idata = az.from_cmdstanpy(fit2)
task1_idata

Inference data with groups:
	> posterior
	> log_likelihood
	> sample_stats

In [54]:
az.summary(task1_idata)

/Users/shenglong/Downloads/vis-decode-analysis/.venv/lib/python3.13/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/Users/shenglong/Downloads/vis-decode-analysis/.venv/lib/python3.13/site-packages/arviz/stats/diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_mod_mean,0.005,0.018,-0.028,0.039,0.000,0.000,19960.0,12381.0,1.0
mu_mod_sd,0.048,0.024,0.002,0.088,0.000,0.000,7693.0,5696.0,1.0
mu_med_mean,0.053,0.052,-0.046,0.148,0.001,0.000,5101.0,9401.0,1.0
mu_med_sd,0.128,0.052,0.017,0.223,0.001,0.001,2557.0,2398.0,1.0
log_sigma_mod_mean,-1.495,0.093,-1.672,-1.323,0.001,0.001,18730.0,13038.0,1.0
...,...,...,...,...,...,...,...,...,...
x_org[155],7.272,0.000,7.272,7.272,0.000,0.000,16000.0,16000.0,NaN
x_org[156],7.608,0.000,7.608,7.608,0.000,0.000,16000.0,16000.0,NaN
x_org[157],7.457,0.000,7.457,7.457,0.000,0.000,16000.0,16000.0,NaN
x_org[158],6.623,0.000,6.623,6.623,0.000,0.000,16000.0,16000.0,NaN


In [ ]:
# TODO: get the posterior 

## `task1.weighted.avg.nonc.normal.stan` 

In [13]:
task1_mix_stanfile = os.path.join('../stan_files/', "task1.weighted.avg.nonc.normal.stan")
task1_model = CmdStanModel(stan_file=task1_mix_stanfile)

16:52:02 - cmdstanpy - INFO - compiling stan file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal.stan to exe file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal
16:52:19 - cmdstanpy - INFO - compiled model executable: /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal


In [14]:
# use shutil to move executable to untracked folder around
dest_dir = "../stan_bin_files"
shutil.move(task1_model.exe_file, os.path.join(dest_dir, task1_model.name))
task1_model = CmdStanModel(exe_file=os.path.join(dest_dir, task1_model.name))

In [15]:
# TODO 

In [ ]:
# TODO 

---

## `task1.mix.stan` 

--- hmm apparently this one doesn't compile because the original definition is a bit wrong ...??? 



In [ ]:
# task1_mix_stanfile = os.path.join('../stan_files/', "task1.weighted.avg.nonc.normal.stan")
# task1_model = CmdStanModel(stan_file=task1_mix_stanfile)

15:02:34 - cmdstanpy - INFO - compiling stan file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal.stan to exe file /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal
15:02:40 - cmdstanpy - INFO - compiled model executable: /Users/shenglong/Downloads/vis-decode-analysis/analysis/stan_files/task1.weighted.avg.nonc.normal
